# CS 3110/5110: Data Privacy
## In-Class Exercise, week of 10/27/2025

In [1]:
# Load the data and libraries
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

def laplace_mech(v, sensitivity, epsilon):
    return v + np.random.laplace(loc=0, scale=sensitivity / epsilon)

def gaussian_mech(v, sensitivity, epsilon, delta):
    return v + np.random.normal(loc=0, scale=sensitivity * np.sqrt(2*np.log(1.25/delta)) / epsilon)

def gaussian_mech_vec(vec, sensitivity, epsilon, delta):
    return [v + np.random.normal(loc=0, scale=sensitivity * np.sqrt(2*np.log(1.25/delta)) / epsilon)
            for v in vec]

def pct_error(orig, priv):
    return np.abs(orig - priv)/orig * 100.0

# adult = pd.read_csv('https://github.com/jnear/cs211-data-privacy/raw/master/homework/adult_with_pii.csv')

In [2]:
# Load data files
import numpy as np
import urllib.request
import io

url_x = 'https://github.com/jnear/cs211-data-privacy/raw/master/slides/adult_processed_x.npy'
url_y = 'https://github.com/jnear/cs211-data-privacy/raw/master/slides/adult_processed_y.npy'

with urllib.request.urlopen(url_x) as url:
    f = io.BytesIO(url.read())
X = np.load(f)

with urllib.request.urlopen(url_y) as url:
    f = io.BytesIO(url.read())
y = np.load(f)

In [3]:
X.shape

(45220, 104)

In [4]:
# Split data into training and test sets
training_size = int(X.shape[0] * 0.8)

X_train = X[:training_size]
X_test = X[training_size:]

y_train = y[:training_size]
y_test = y[training_size:]

print('Train and test set sizes:', len(y_train), len(y_test))

Train and test set sizes: 36176 9044


## Question 1

Using scikit-learn, train a logistic regression model on the training data loaded above.

In [5]:
from sklearn.linear_model import LogisticRegression

In [6]:
def train_model():
    model = LogisticRegression()
    model.fit(X_train, y_train)
    return model

model = train_model()
print('Model coefficients:', model.coef_[0])
print('Model accuracy:', np.sum(model.predict(X_test) == y_test)/X_test.shape[0])

Model coefficients: [ 3.22958884e-01 -3.57306786e-01 -1.85450976e-01 -8.06914689e-02
 -7.39295459e-01 -5.27268959e-01 -9.29960595e-01 -7.74714549e-01
 -7.32356344e-01 -5.30503794e-01 -1.13074464e+00 -5.65991637e-01
 -8.59361278e-01 -8.89811173e-01  5.13122941e-02  8.49360109e-02
  5.13313048e-01  1.04764159e+00 -1.76361459e-01  7.64755657e-01
 -6.51767519e-01  1.29011170e+00  6.25267352e-02 -1.02809816e+00
  1.35034987e+00  1.22888767e+00 -7.10258162e-01 -1.43669446e+00
 -1.01748435e+00 -8.83717759e-01 -8.37513307e-02 -1.03160596e-02
 -3.21459189e-02  6.88679383e-01 -9.40116079e-01 -7.35491693e-01
 -3.63520836e-01 -9.45370640e-01 -1.45310889e+00  4.44083098e-01
  4.60628168e-01  1.87508859e-01  4.61468442e-01 -1.75561867e-01
 -5.68973783e-01 -8.68334973e-02 -9.23221803e-01 -1.15708883e+00
 -2.76079898e-01  5.15182450e-01 -6.78048806e-01 -1.74817689e-01
 -6.68065260e-01 -5.67879790e-01 -4.08203815e-01 -1.58428676e+00
 -9.12728599e-01  8.58290128e-01  6.67071803e-01 -3.23352900e-01
 -1.3

## Question 2

Implement the *average gradient* of the loss below.

In [7]:
# The loss function measures how good our model is. The training goal is to minimize the loss.
# This is the logistic loss function.
def loss(theta, xi, yi):
    exponent = - yi * (xi.dot(theta))
    return np.log(1 + np.exp(exponent))

# This is the gradient of the logistic loss
# The gradient is a vector that indicates the rate of change of the loss in each direction
# gradient tells us hot to chege thetta to make loss  bigger
def gradient(theta, xi, yi):
    exponent = yi * (xi.dot(theta))
    return - (yi*xi) / (1+np.exp(exponent))

In [8]:
def avg_grad(theta, X, y):
    grads = [gradient(theta, x_i, y_i) for x_i, y_i in zip(X,y)]
    avg_grad = np.mean(grads, axis=0)
    return avg_grad


## Question 3

Use the average gradient from above to implement a gradient descent algorithm.

In [9]:
def gradient_descent(iterations):
    theta = np.zeros(X_train.shape[1])
    for i in range(iterations):
        theta = theta - avg_grad(theta, X_train, y_train)
    return theta

theta = gradient_descent(10)


In [10]:
# Prediction: take a model (theta) and a single example (xi) and return its predicted label
def predict(xi, theta, bias=0):
    label = np.sign(xi @ theta + bias)
    return label

def accuracy(theta):
    return np.sum(predict(X_test, theta) == y_test)/X_test.shape[0]

accuracy(theta)

np.float64(0.7787483414418399)

## Question 4

Implement a *noisy gradient descent* algorithm.

1. Calculate gradients for each example
2. Clip the gradients to have bounded $L2$ norm
3. Sum the clipped gradients
4. Use the Gaussian mechanism to add noise to the sum of gradients

In [25]:
def L2_clip(v, b):
    norm = np.linalg.norm(v, ord=2)
    
    if norm > b:
        return b * (v / norm)
    else:
        return v

def noisy_gradient_descent(iterations, epsilon, delta):
    theta = np.zeros(X_train.shape[1])
    for i in range(iterations):
        grads = [gradient(theta, x_i, y_i) for x_i, y_i in zip(X_train,y_train)]
        # TODO: :clipping to bound L2 sensitivity of the sum
        b = 3
        clipped_grads = [L2_clip(g,b) for g in grads]
        sum_grad = np.sum(clipped_grads, axis=0)
        # TODO: that is wrong
        noisy_sum = gaussian_mech_vec(sum_grad, sensitivity=b, epsilon=epsilon, delta=delta)
        noisy_count = laplace_mech(len(X_train), sensitivity=1, epsilon=epsilon)
        noisy_grad = np.array(noisy_sum) / noisy_count
        theta = theta - noisy_grad
    return theta

theta = noisy_gradient_descent(10, 10.0, 1e-5)
print('Final accuracy:', accuracy(theta))

Final accuracy: 0.7787483414418399


In [22]:
# TEST CASE

assert accuracy(noisy_gradient_descent(5, 0.001, 1e-5)) < 0.76
assert accuracy(noisy_gradient_descent(5, 1.0, 1e-5)) > 0.70

## Question 5

What is the *total privacy cost* of the noisy gradient descent algorithm above, and why? Argue informally that the algorithm satisfies this privacy cost. Use sequential composition.

1. Sensitivity:  
    - The laplace mechanism has a sensitivity of 1 beacause it's countign query
    - The use of the vector gaussian mech has L2 sensitivity of b because I clip each gradient to have L2 norm bounded by b and then sum the clipped gradients
2. Composition: total proivacy cost is 'iterations*epsilon', 'iterations*delta' by sequential composition
3. Post-processing: 
    1. I return the final theta
    2. Each is computed using the prior theta and the (differentional private) noisy gradient
    3. The initial theta is constant (all 0)
    4. So final theta satisfies differential privacy by post processing

## Question 6

Repeat the above, but using advanced composition.

1. Sensitivity:  
    - The laplace mechanism has a sensitivity of 1 beacause it's countign query
    - The use of the vector gaussian mech has L2 sensitivity of b because I clip each gradient to have L2 norm bounded by b and then sum the clipped gradients
2. Composition: 
    - total privacy cost for the loop is:
        - `epsilon'`, `delta' + iterations*delta` where
        - `epsilon'` = `epsilon * sqrt(2*iterations*log(1/delta')) + iterations*epsilon (e^epsilon -1)`
    - total proivacy cost for the function is `epsilon' + epsilon`, `delta + iterations*delta`  by sequential composition
3. Post-processing: 
    1. I return the final theta
    2. Each is computed using the prior theta and the (differentional private) noisy gradient
    3. The initial theta is constant (all 0)
    4. So final theta satisfies differential privacy by post processing

## Question 7

Implement a version of noisy gradient descent that satisfies a *total* of $(\epsilon, \delta)$-differential privacy. Use sequential composition.

In [31]:
def noisy_gradient_descent(iterations, epsilon, delta):
    epsilon_i = epsilon/(iterations+1)
    delta_i = delta/iterations
    theta = np.zeros(X_train.shape[1])
    for i in range(iterations):
        grads = [gradient(theta, x_i, y_i) for x_i, y_i in zip(X_train,y_train)]
        # TODO: :clipping to bound L2 sensitivity of the sum
        b = 3
        clipped_grads = [L2_clip(g,b) for g in grads]
        sum_grad = np.sum(clipped_grads, axis=0)
        # TODO: that is wrong
        noisy_sum = gaussian_mech_vec(sum_grad, sensitivity=b, epsilon=epsilon_i, delta=delta_i)
        noisy_count = laplace_mech(len(X_train), sensitivity=1, epsilon=epsilon_i)
        noisy_grad = np.array(noisy_sum) / noisy_count
        theta = theta - noisy_grad
    return theta

accuracy(theta)

np.float64(0.7787483414418399)

In [29]:
# TEST CASE

assert accuracy(noisy_gradient_descent(5, 0.001, 1e-5)) < 0.76
assert accuracy(noisy_gradient_descent(5, 1.0, 1e-5)) > 0.70